# Phase 2 — E4: Tied embeddings + label smoothing + warmup-cosine

In [1]:
import sys
from pathlib import Path
nb_dir = Path('.').resolve()
sys.path.insert(0, str(nb_dir / "output"))
import importlib, exp_runner
importlib.reload(exp_runner)
from exp_runner import ExperimentConfig, run_experiment, evaluate_only
print("exp_runner loaded from", exp_runner.__file__)


exp_runner loaded from /home/coezbek/dev/2026/AT3_training/assignment3_notebook/output/exp_runner.py


Three small training tricks bundled together:
* tie input embedding with output projection (`tie_embeddings=True`)
* label smoothing 0.1 in the cross-entropy loss
* warmup + cosine LR schedule (1000 step warmup, cosine decay)

Everything else identical to E0. Evaluated with beam search using the best (beam, length_penalty) from E1.

In [2]:
import pandas as pd
e1_sweep = pd.read_csv('output/phase2_results/phase2_e1_beam_sweep.csv')
e1_sweep = e1_sweep.sort_values('test_BLEU-4', ascending=False)
best_beam = int(e1_sweep.iloc[0]['beam_width']) if 'beam_width' in e1_sweep.columns else 5
best_lp = float(e1_sweep.iloc[0]['length_penalty']) if 'length_penalty' in e1_sweep.columns else 1.0
print('Using beam_width=', best_beam, ' length_penalty=', best_lp)

cfg = ExperimentConfig(
    run_name='phase2_e4_schedule_polish',
    output_dir='output/phase2_results',
    epochs=15, batch_size=128, lr=1e-4,
    optimizer='adamw', weight_decay=0.01,
    scheduler='warmup_cosine', warmup_steps=500,
    label_smoothing=0.1, tie_embeddings=True,
    decoding='beam', beam_width=best_beam, length_penalty=best_lp,
)
metrics_e4 = run_experiment(cfg)
metrics_e4

Using beam_width= 5  length_penalty= 0.7
[phase2_e4_schedule_polish] device=cuda


(null): No such file or directory


/home/coezbek/dev/2026/AT3_training/.venv/lib/python3.13/site-packages/torch/nn/functional.py:6682: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:383.)
  attn_output = scaled_dot_product_attention(
/home/coezbek/dev/2026/AT3_training/.venv/lib/python3.13/site-packages/torch/nn/functional.py:6682: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:323.)
  attn_output = scaled_dot_product_attention(


[phase2_e4_schedule_polish] ep01/15 train=7.2830  val=5.3713  lr=4.14e-05  t=43.6s


[phase2_e4_schedule_polish] ep02/15 train=5.0369  val=4.6519  lr=8.28e-05  t=44.0s


...

[phase2_e4_schedule_polish] ep13/15 train=3.4243  val=3.9551  lr=6.10e-06  t=44.0s


[phase2_e4_schedule_polish] ep14/15 train=3.4094  val=3.9552  lr=1.55e-06  t=44.1s


[phase2_e4_schedule_polish] ep15/15 train=3.4058  val=3.9551  lr=0.00e+00  t=44.2s


[phase2_e4_schedule_polish] DONE  val_BLEU-4=0.3452  test_BLEU-4=0.3618  test_CIDEr=1.3931


{'run_name': 'phase2_e4_schedule_polish',
 'best_val_loss': 3.955068408118354,
 'epochs_run': 15,
 'decoding': 'beam',
 'beam_width': 5,
 'length_penalty': 0.7,
 'val_BLEU-1': 0.5364985128265046,
 'val_BLEU-2': 0.44474574438547615,
 'val_BLEU-3': 0.38527981712923437,
 'val_BLEU-4': 0.3452371342013412,
 'val_CIDEr': 1.3093260592317946,
 'test_BLEU-1': 0.5459991045103828,
 'test_BLEU-2': 0.4550799591773138,
 'test_BLEU-3': 0.3988031966246516,
 'test_BLEU-4': 0.36179532051134944,
 'test_CIDEr': 1.3931053088927083}